# BiLSTM Speech Enhancement - Colab training

**Before running:** set the runtime to **GPU** (Runtime -> Change runtime type -> T4 GPU).

One-time dataset prep on your PC (see `scripts/prepare_dataset.md`):
```
cd <repo>
zip -r voicebank_demand.zip speech/clean_trainset_wav speech/noisy_trainset_wav \
                            speech/clean_testset_wav  speech/noisy_testset_wav
```
then upload `voicebank_demand.zip` (~2.6 GB) to the root of your Google Drive (`MyDrive/`).

In [ ]:
# 1. Clone the rebuild branch
!git clone --branch bilstm-rebuild https://github.com/AsimShareef/BiLSTM-Speech-Enhancement-System.git repo
%cd repo
!git log --oneline -3

In [ ]:
# 2. Dependencies (Colab already has tensorflow + numpy/scipy/librosa)
!pip -q install pystoi pesq
import tensorflow as tf
print('TF', tf.__version__, '| GPU:', tf.config.list_physical_devices('GPU'))

In [ ]:
# 3. Pull the dataset from Drive and unzip to ./speech
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p speech
!unzip -q -o /content/drive/MyDrive/voicebank_demand.zip -d .
!ls speech && echo '---' && ls speech/clean_trainset_wav | wc -l

In [ ]:
# 4. Sanity-check the streaming dataset (no training yet)
!python src/dataset.py

In [ ]:
# 5. Train on the full corpus (early stopping usually halts well before 40 epochs)
!python src/train.py --epochs 40 --batch 128

In [ ]:
# 6. Evaluate on all 824 test files: BiLSTM vs spectral-subtraction vs noisy
!python src/evaluate.py --model bilstm_enhancer.keras
print(open('results/metrics.md').read())

In [ ]:
# 7. Save artefacts back to Drive (model + metrics + sample clips)
!mkdir -p /content/drive/MyDrive/bilstm_se_out
!cp bilstm_enhancer.keras checkpoints/history.csv /content/drive/MyDrive/bilstm_se_out/
!cp -r results /content/drive/MyDrive/bilstm_se_out/
print('done - download bilstm_se_out/ from Drive and commit results/ + attach the .keras to a GitHub Release')